<a href="https://colab.research.google.com/drive/1xWtKdjvrUGjjm6w20Ssjc6EB1I9wMu2i?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Model Context Protocol (MCP)

In [ ]:
!pip install -qU google-genai

In [ ]:
from google import genai
import getpass

Get free-tier Google's Gemini API Key here: https://aistudio.google.com/app/apikey

In [ ]:
API_KEY = getpass.getpass("Enter your Google API key: ")

In [ ]:
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-flash-latest"

In [ ]:
import json

In [ ]:
class MCPServer:
    """A minimal MCP server, speaking the real message shapes in-process.

    A production MCP server runs as a separate process and exchanges these
    same JSON-RPC messages over stdio or HTTP. Everything protocol-shaped
    here is real; only the transport is a function call, so the notebook
    needs no second process and no network.
    """

    PROTOCOL_VERSION = "2025-06-18"

    def __init__(self, name):
        self.name = name
        self._tools = {}
        self.log = []

    def tool(self, description, schema):
        """Register a Python function as an MCP tool."""
        def register(func):
            self._tools[func.__name__] = {
                "name": func.__name__,
                "description": description,
                "inputSchema": schema,
                "func": func,
            }
            return func
        return register

    def handle(self, request):
        """The entire server: one JSON-RPC request in, one response out."""
        self.log.append(request)
        method = request.get("method")
        request_id = request.get("id")

        if method == "initialize":
            return self._ok(request_id, {
                "protocolVersion": self.PROTOCOL_VERSION,
                "capabilities": {"tools": {}},
                "serverInfo": {"name": self.name, "version": "1.0.0"},
            })

        if method == "tools/list":
            return self._ok(request_id, {
                "tools": [
                    {k: v for k, v in tool.items() if k != "func"}
                    for tool in self._tools.values()
                ]
            })

        if method == "tools/call":
            params = request.get("params", {})
            tool = self._tools.get(params.get("name"))
            if tool is None:
                # -32602 is JSON-RPC "invalid params". An unknown tool is a
                # protocol-level error, not a tool result.
                return self._error(request_id, -32602,
                                   f"Unknown tool: {params.get('name')}")
            try:
                result = tool["func"](**params.get("arguments", {}))
            except Exception as exc:
                # A tool that raises is NOT a protocol error. It comes back as
                # a normal result with isError set, so the model can read the
                # message and try something else instead of the session dying.
                return self._ok(request_id, {
                    "content": [{"type": "text", "text": f"{type(exc).__name__}: {exc}"}],
                    "isError": True,
                })
            return self._ok(request_id, {
                "content": [{"type": "text", "text": str(result)}],
                "isError": False,
            })

        return self._error(request_id, -32601, f"Method not found: {method}")

    @staticmethod
    def _ok(request_id, result):
        return {"jsonrpc": "2.0", "id": request_id, "result": result}

    @staticmethod
    def _error(request_id, code, message):
        return {"jsonrpc": "2.0", "id": request_id,
                "error": {"code": code, "message": message}}

In [ ]:
server = MCPServer("notebook-demo")

INVENTORY = {"SKU-100": 42, "SKU-200": 0, "SKU-300": 7}
PRICES = {"SKU-100": 19.99, "SKU-200": 249.00, "SKU-300": 5.50}


@server.tool(
    description="Return the number of units in stock for a SKU.",
    schema={
        "type": "object",
        "properties": {"sku": {"type": "string", "description": "e.g. SKU-100"}},
        "required": ["sku"],
    },
)
def check_stock(sku):
    if sku not in INVENTORY:
        raise KeyError(f"no such SKU: {sku}")
    return INVENTORY[sku]


@server.tool(
    description="Return the total price for a quantity of a SKU.",
    schema={
        "type": "object",
        "properties": {
            "sku": {"type": "string"},
            "quantity": {"type": "integer", "minimum": 1},
        },
        "required": ["sku", "quantity"],
    },
)
def quote_price(sku, quantity):
    if sku not in PRICES:
        raise KeyError(f"no such SKU: {sku}")
    return round(PRICES[sku] * quantity, 2)


handshake = server.handle({"jsonrpc": "2.0", "id": 1, "method": "initialize"})
print("initialize ->")
print(json.dumps(handshake, indent=2))

listing = server.handle({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
print("\ntools/list ->")
print(json.dumps(listing, indent=2))

In [ ]:
class MCPClient:
    """The host side: discover tools, expose them to the model, relay calls.

    The client never hardcodes a tool. It asks the server what exists and
    hands that straight to the model - which is the entire point of MCP. Add
    a tool to the server and this class does not change.
    """

    def __init__(self, server):
        self.server = server
        self.next_id = 100
        self.tools = []

    def _send(self, method, params=None):
        self.next_id += 1
        request = {"jsonrpc": "2.0", "id": self.next_id, "method": method}
        if params is not None:
            request["params"] = params
        return self.server.handle(request)

    def connect(self):
        self._send("initialize")
        self.tools = self._send("tools/list")["result"]["tools"]
        return self.tools

    def as_gemini_declarations(self):
        """MCP inputSchema is JSON Schema, which is what Gemini wants too."""
        return [{
            "name": tool["name"],
            "description": tool["description"],
            "parameters": tool["inputSchema"],
        } for tool in self.tools]

    def call(self, name, arguments):
        response = self._send("tools/call", {"name": name, "arguments": arguments})
        if "error" in response:
            return f"protocol error: {response['error']['message']}"
        result = response["result"]
        text = "".join(part["text"] for part in result["content"])
        return f"tool error: {text}" if result.get("isError") else text

    def ask(self, question, max_steps=5):
        """Let the model drive the tools until it has an answer."""
        print(f"\nQuestion: {question}\n")

        contents = [{"role": "user", "parts": [{"text": question}]}]
        config = {"tools": [{"function_declarations": self.as_gemini_declarations()}]}

        for step in range(max_steps):
            response = client.models.generate_content(
                model=MODEL_NAME, contents=contents, config=config
            )
            candidate = response.candidates[0]
            calls = [p.function_call for p in candidate.content.parts if p.function_call]

            if not calls:
                answer = "".join(p.text for p in candidate.content.parts if p.text)
                print(f"Answer: {answer.strip()}")
                return answer.strip()

            contents.append(candidate.content)
            replies = []
            for fc in calls:
                arguments = dict(fc.args)
                result = self.call(fc.name, arguments)
                print(f"  step {step + 1}: {fc.name}({arguments}) -> {result}")
                replies.append({"function_response": {"name": fc.name,
                                                      "response": {"result": result}}})
            contents.append({"role": "user", "parts": replies})

        return "gave up after the step limit"

In [ ]:
mcp = MCPClient(server)
discovered = mcp.connect()
print(f"Discovered {len(discovered)} tools without knowing any of them in advance:")
for tool in discovered:
    print(f"  - {tool['name']}: {tool['description']}")

print("\n" + "=" * 60)
print("EXAMPLE 1: One tool")
print("=" * 60)
mcp.ask("How many units of SKU-100 are in stock?")

print("\n" + "=" * 60)
print("EXAMPLE 2: Two tools, chained")
print("=" * 60)
mcp.ask("If I buy every SKU-300 unit you have in stock, what does it cost?")

print("\n" + "=" * 60)
print("EXAMPLE 3: A tool that fails")
print("=" * 60)
mcp.ask("How many units of SKU-999 are in stock?")

In [ ]:
# Every message that crossed the boundary
#
# This is the whole value of the protocol being a protocol: the exchange is
# inspectable, and it would look identical over stdio to a server written in
# TypeScript, Rust, or anything else.

print(f"{len(server.log)} JSON-RPC requests reached the server\n")
for request in server.log:
    params = request.get("params", "")
    print(f"  {request['method']:12s} {params if params else ''}")

In [ ]:
# What a real deployment adds
#
# - Transport. A real server runs as its own process over stdio or
#   streamable HTTP. The message shapes above do not change; only `handle`
#   is replaced by a pipe or a socket.
# - Trust. A tool description is untrusted text from another program, and it
#   goes straight into the model's context. A malicious server can put
#   instructions there - so treat descriptions as data, and gate anything
#   destructive behind human approval rather than the model's judgement.
# - Resources and prompts. Tools are one of three MCP primitives; servers can
#   also expose readable resources and reusable prompt templates.
#
# Rather than writing the client by hand, production code uses the official
# `mcp` package - the point of this notebook is that you now know exactly
# what that package is doing underneath.

print("Protocol version spoken above:", MCPServer.PROTOCOL_VERSION)